# European Streetview CNN — v3 (Upgraded)
Improvements over v2 (ResNet50):

| What | Why |
|---|---|
| **EfficientNet-B3 @ 384 px** | Reads sign text; stronger features than ResNet50 |
| **Weighted random sampler** | Fixes class imbalance — small countries trained fairly |
| **Heavy augmentation (albumentations)** | Perspective warp, grid distort, blur, cutout, RandAugment |
| **Sign-crop auxiliary loss** | Forces backbone to learn sign/plate/text features explicitly |
| **Script-detection feature vector** | Free high-signal Cyrillic/Greek/Latin cue concatenated into head |
| **Multi-task heads** | Aux head predicts language group — guides backbone |
| **Test-time augmentation (TTA)** | 5-crop + h-flip ensemble at inference — free +1–3% |
| **GradCAM inspection cell** | See what the model actually looks at |
| **Confusion matrix hard-pair mining** | Identify worst country pairs; over-sample them |


## 0 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil, time, os

_DRIVE_TRAIN = '/content/drive/MyDrive/GeoGussrCheat/trainEurope'
_DRIVE_TEST  = '/content/drive/MyDrive/GeoGussrCheat/testEurope'
_LOCAL_TRAIN = '/content/trainEurope'
_LOCAL_TEST  = '/content/testEurope'

def _copy_if_needed(src, dst):
    if os.path.exists(dst):
        print(f'  Already exists, skipping: {dst}')
        return
    print(f'  Copying {src} → {dst} ...', end=' ', flush=True)
    t0 = time.time()
    shutil.copytree(src, dst)
    n = sum(len(f) for _, _, f in os.walk(dst))
    print(f'{n} files copied in {time.time()-t0:.0f}s')

print('Copying dataset to local SSD...')
_copy_if_needed(_DRIVE_TRAIN, _LOCAL_TRAIN)
_copy_if_needed(_DRIVE_TEST,  _LOCAL_TEST)
print('Done.')


## 1 · Install extra dependencies

In [ ]:
# albumentations: heavy augmentation pipeline
# grad-cam: GradCAM visualisation
# paddleocr: real script/language detection from sign text
!pip install -q albumentations grad-cam tqdm
!pip install -q paddlepaddle-gpu paddleocr   # GPU build — faster on Colab
print('Done.')


## 2 · Imports & reproducibility

In [ ]:
import os, random, json, time, warnings
from collections import Counter
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 3 · Configuration

In [ ]:
TRAIN_DIR = '/content/trainEurope'
TEST_DIR  = '/content/testEurope'

IMG_SIZE        = 384
BATCH_SIZE      = 32
NUM_EPOCHS      = 50
FREEZE_EPOCHS   = 5
LR_HEAD         = 5e-4
LR_FULL         = 5e-5
VAL_SPLIT       = 0.20
NUM_WORKERS     = min(os.cpu_count() or 2, 4)

LAMBDA_SIGN     = 0.30
LAMBDA_LANG     = 0.15

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── 16-dimensional script/diacritic feature vector
# Each dimension is a distinct, highly discriminative writing signal.
# Index → feature name → countries it fires for
#
#  0  Cyrillic          Bulgaria, Serbia, Montenegro, North Macedonia, Bosnia (some), Ukraine, Belarus
#  1  Greek             Greece, Cyprus
#  2  Georgian          Georgia
#  3  Armenian          Armenia
#  4  Arabic            Bosnia (minority), Kosovo (minority)
#  5  Hebrew            Israel (if dataset ever expands)
#  6  Polish            Poland         (ą ę ó ź ż ń ć ś)
#  7  Hungarian         Hungary        (ő ű + á é í ó ö ú ü)
#  8  Romanian          Romania        (ș ț â î)
#  9  Czech/Slovak      Czech Rep, Slovakia  (č š ž ř ě ď ť)
# 10  Scandinavian      Norway, Sweden, Denmark, Finland (å ø æ ö)
# 11  Icelandic/Faroese Iceland, Faroe Islands (þ ð æ)
# 12  Turkish           Turkey         (ğ ş ı İ)
# 13  Baltic            Latvia, Lithuania, Estonia (ā ē ī ū č š ž ģ ķ ļ ņ)
# 14  Maltese           Malta          (ħ ġ)
# 15  Plain Latin       all other Latin-script countries (fallback)

NUM_LANG_FEATURES = 16

# Ground-truth mapping: country folder name → list of feature indices that are active
# Used ONLY during training (where label is known).
# At inference, detect_script_from_image() derives this from the actual image.
COUNTRY_LANG_FEATURES = {
    # Cyrillic countries (idx 0)
    'Bulgaria':        [0],
    'Serbia':          [0],
    'Montenegro':      [0],
    'North Macedonia': [0],
    'Ukraine':         [0],
    'Belarus':         [0],
    'Russia':          [0],
    'Bosnia and Herzegovina': [0, 15],   # dual-script (Cyrillic + Latin)

    # Greek (idx 1)
    'Greece':  [1],
    'Cyprus':  [1],

    # Georgian (idx 2)
    'Georgia': [2],

    # Armenian (idx 3)
    'Armenia': [3],

    # Turkish (idx 12)
    'Turkey': [12],

    # Polish (idx 6)
    'Poland': [6],

    # Hungarian (idx 7)
    'Hungary': [7],

    # Romanian (idx 8)
    'Romania': [8],

    # Czech / Slovak (idx 9)
    'Czech Republic': [9],
    'Czechia':        [9],
    'Slovakia':       [9],

    # Scandinavian (idx 10)
    'Sweden':  [10],
    'Norway':  [10],
    'Denmark': [10],
    'Finland': [10],

    # Icelandic / Faroese (idx 11)
    'Iceland':        [11],
    'Faroe Islands':  [11],

    # Baltic (idx 13)
    'Latvia':    [13],
    'Lithuania': [13],
    'Estonia':   [13],

    # Maltese (idx 14)
    'Malta': [14],

    # Plain Latin fallback (idx 15) — everything not listed above
}

def lang_multihot(country_name):
    """
    Multi-hot 16-dim float32 vector from country name.
    Used ONLY during training where the label is already known.
    At inference, use detect_script_from_image() instead.
    """
    v = np.zeros(NUM_LANG_FEATURES, dtype=np.float32)
    indices = COUNTRY_LANG_FEATURES.get(country_name, [15])   # default: plain Latin
    for idx in indices:
        v[idx] = 1.0
    return v

FEATURE_NAMES = [
    'Cyrillic', 'Greek', 'Georgian', 'Armenian', 'Arabic', 'Hebrew',
    'Polish', 'Hungarian', 'Romanian', 'Czech/Slovak',
    'Scandinavian', 'Icelandic/Faroese', 'Turkish', 'Baltic', 'Maltese',
    'Plain Latin',
]

print(f'IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, NUM_EPOCHS={NUM_EPOCHS}')
print(f'Language feature dimensions: {NUM_LANG_FEATURES}')
print('Features:', FEATURE_NAMES)


## 4 · Augmentation pipelines

We use **albumentations** for richer augmentation than torchvision: perspective warp, grid distortion, blur variants, coarse dropout (cutout), and brightness/contrast — all things that help the model generalise across different camera setups, weather, and seasons.

In [ ]:
# NOTE: written for albumentations 2.x API (Colab default).
#   - RandomResizedCrop uses size=(h, w)  (not height=/width=)
#   - ImageCompression uses quality_range=(lo, hi)
#   - GaussNoise uses std_range=(lo, hi)   (fraction of 255, not variance)
#   - CoarseDropout uses num_holes_range / hole_height_range / hole_width_range / fill
#   - Affine replaces the deprecated ShiftScaleRotate

def build_train_transform(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size),
                            scale=(0.6, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(p=0.5),

        # ── Geometric distortions (simulate different camera/lens)
        A.OneOf([
            A.Perspective(scale=(0.02, 0.07), p=1.0),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
            A.OpticalDistortion(distort_limit=0.1, p=1.0),
        ], p=0.4),

        A.Affine(translate_percent=(0.0, 0.05), scale=(0.9, 1.1),
                 rotate=(-10, 10), border_mode=cv2.BORDER_REFLECT, p=0.5),

        # ── Colour / weather simulation
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20),
            A.CLAHE(clip_limit=3.0),
        ], p=0.6),

        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=7),
            A.MedianBlur(blur_limit=5),
        ], p=0.3),

        A.ImageCompression(quality_range=(60, 100), p=0.2),
        A.GaussNoise(std_range=(0.04, 0.13), p=0.2),

        # ── Cutout: hides patches so model can't rely on a single region
        A.CoarseDropout(num_holes_range=(1, 6),
                        hole_height_range=(8, img_size // 8),
                        hole_width_range=(8, img_size // 8),
                        fill=0, p=0.3),

        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_transform(img_size=384):
    return A.Compose([
        A.Resize(height=int(img_size * 1.15), width=int(img_size * 1.15)),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

# Sign-crop transform: smaller, tighter, no geometric distortion
def build_sign_transform(size=160):
    return A.Compose([
        A.Resize(height=size, width=size),
        A.HorizontalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

train_tf = build_train_transform(IMG_SIZE)
val_tf   = build_val_transform(IMG_SIZE)
sign_tf  = build_sign_transform(160)
print('Transforms built (albumentations', A.__version__, ').')


## 5 · Sign-crop helper

For every full image we extract **two random upper-third crops** (where signs typically live in streetview images). These crops get their own forward pass through a shared auxiliary head during training, contributing 30% of the total loss. This forces the backbone to learn sign-specific features.

In [ ]:
def extract_sign_crops(imgs_np, sign_transform, n_crops=2):
    """
    imgs_np : list of H×W×3 uint8 numpy arrays (original images, before normalisation)
    Returns a (B*n_crops, 3, H_s, W_s) tensor of sign-region crops.

    Strategy: take n_crops random crops from the upper 40% of the image
    (signs, text, and road markings are most dense there in streetview).
    """
    crops = []
    for img in imgs_np:
        h, w = img.shape[:2]
        upper_h = max(int(h * 0.40), 80)
        for _ in range(n_crops):
            # random crop within upper region
            y0 = random.randint(0, max(0, upper_h - 80))
            x0 = random.randint(0, max(0, w - 160))
            y1 = min(y0 + random.randint(80, upper_h), h)
            x1 = min(x0 + random.randint(120, w // 2), w)
            crop = img[y0:y1, x0:x1]
            if crop.size == 0:
                crop = img[:upper_h, :w//2]
            aug = sign_transform(image=crop)['image']
            crops.append(aug)
    return torch.stack(crops)   # (B*n_crops, 3, 160, 160)


## 6 · Script-detection feature vector

A lightweight heuristic (no OCR needed) encodes **which writing script** is visible. Cyrillic and Greek presence is an extremely strong signal for country classification. We concatenate a 3-dim one-hot into the classifier head alongside CNN features.

In [ ]:
# ── Full 16-dim script + diacritic detection via PaddleOCR
# At TRAINING time: lang_multihot(country_name) gives ground-truth.
# At INFERENCE time: detect_script_from_image() reads actual pixels.

from paddleocr import PaddleOCR
import unicodedata, re

_ocr_engine = None

def get_ocr_engine():
    global _ocr_engine
    if _ocr_engine is None:
        print('Initialising PaddleOCR (first call only)...')
        _ocr_engine = PaddleOCR(
            use_angle_cls=True,
            lang='en',       # multilingual detection backbone
            use_gpu=True,
            show_log=False,
        )
        print('PaddleOCR ready.')
    return _ocr_engine


# ── Unicode range patterns for non-Latin scripts
_RANGES = {
    0:  re.compile(r'[Ѐ-ӿ]'),   # Cyrillic
    1:  re.compile(r'[Ͱ-Ͽ]'),   # Greek
    2:  re.compile(r'[Ⴀ-ჿ]'),   # Georgian
    3:  re.compile(r'[԰-֏]'),   # Armenian
    4:  re.compile(r'[؀-ۿ]'),   # Arabic
    5:  re.compile(r'[֐-׿]'),   # Hebrew
}

# ── Latin diacritic fingerprints (characters unique to each language group)
_DIACRITICS = {
    6:  set('ąęóźżńćśĄĘÓŹŻŃĆŚ'),                        # Polish
    7:  set('őűŐŰáéíóöúüÁÉÍÓÖÚÜ'),                      # Hungarian
    8:  set('șțâîȘȚÂÎ'),                                  # Romanian
    9:  set('čšžřěďťČŠŽŘĚĎŤ'),                           # Czech/Slovak
    10: set('åøæöÅØÆÖ'),                                   # Scandinavian
    11: set('þðÞÐ'),                                        # Icelandic/Faroese
    12: set('ğşıİĞŞ'),                                      # Turkish
    13: set('āēīūčšžģķļņĀĒĪŪČŠŽĢĶĻŅ'),                   # Baltic
    14: set('ħġĦĠ'),                                        # Maltese
}


def _score_text(all_text):
    """
    Given all OCR-detected text concatenated, return raw character counts
    for each of the 16 feature dimensions.
    """
    counts = np.zeros(NUM_LANG_FEATURES, dtype=np.float32)

    for feat_idx, pattern in _RANGES.items():
        counts[feat_idx] = len(pattern.findall(all_text))

    for feat_idx, char_set in _DIACRITICS.items():
        counts[feat_idx] = sum(1 for c in all_text if c in char_set)

    # Plain Latin: Latin chars not caught by any diacritic group
    all_diac = set().union(*_DIACRITICS.values())
    counts[15] = sum(
        1 for c in all_text
        if unicodedata.category(c).startswith('L')
        and not any(p.match(c) for p in _RANGES.values())
        and c not in all_diac
    )

    return counts


def detect_script_from_image(img_input, confidence_threshold=0.5):
    """
    img_input : file path (str) OR numpy H×W×3 RGB array
    Returns   : np.float32 array of shape (16,) — soft confidence scores,
                one per script/diacritic feature dimension.

    The vector is L1-normalised so values sum to 1 (interpretable as
    proportions of character evidence). If no text is detected, returns
    a plain-Latin default [0,0,...,0,1] (index 15 = 1.0).
    """
    import cv2
    ocr = get_ocr_engine()

    if isinstance(img_input, str):
        result = ocr.ocr(img_input, cls=True)
    else:
        img_bgr = cv2.cvtColor(img_input, cv2.COLOR_RGB2BGR)
        result  = ocr.ocr(img_bgr, cls=True)

    # Collect all high-confidence text
    all_text = ''
    if result and result[0]:
        for line in result[0]:
            text, conf = line[1][0], line[1][1]
            if conf >= confidence_threshold:
                all_text += text + ' '

    if not all_text.strip():
        # No text detected — return plain Latin default
        v = np.zeros(NUM_LANG_FEATURES, dtype=np.float32)
        v[15] = 1.0
        return v

    counts = _score_text(all_text)
    total  = counts.sum() + 1e-8
    return counts / total   # L1-normalised soft vector


def explain_script_detection(img_input):
    """
    Human-readable breakdown of what scripts/diacritics were detected.
    Useful for debugging — call this in the GradCAM / analysis section.
    """
    vec = detect_script_from_image(img_input)
    print('Script detection breakdown:')
    for i, (name, score) in enumerate(zip(FEATURE_NAMES, vec)):
        bar = '█' * int(score * 40)
        print(f'  {i:2d}  {name:20s}  {score:.3f}  {bar}')
    top = vec.argmax()
    print(f'\n  → Dominant: {FEATURE_NAMES[top]} ({vec[top]:.1%})')
    return vec


print('Script detection ready — 16 features:')
for i, name in enumerate(FEATURE_NAMES):
    print(f'  {i:2d}  {name}')


## 7 · Dataset

In [ ]:
class StreetviewDataset(Dataset):
    """
    Loads images from root/<country>/<img>.jpg
    Returns: (path, label_idx, lang_vec) — lang_vec is 3-dim script one-hot.
    """
    def __init__(self, root_dir, transform=None, class_to_idx=None,
                 sign_transform=None, return_numpy=False):
        self.root_dir       = root_dir
        self.transform      = transform
        self.sign_transform = sign_transform
        self.return_numpy   = return_numpy  # return raw np array for sign crops
        self.samples        = []  # (path, label_idx, lang_vec)

        countries = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.class_to_idx = class_to_idx or {c: i for i, c in enumerate(countries)}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.classes = countries

        IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
        for country in countries:
            if country not in self.class_to_idx:
                continue
            label = self.class_to_idx[country]
            lv    = lang_multihot(country)
            folder = os.path.join(root_dir, country)
            for fname in os.listdir(folder):
                if os.path.splitext(fname)[1].lower() in IMG_EXTS:
                    self.samples.append((os.path.join(folder, fname), label, lv))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, lang_vec = self.samples[idx]
        img_pil = Image.open(path).convert('RGB')
        img_np  = np.array(img_pil)

        if self.transform:
            img_t = self.transform(image=img_np)['image']
        else:
            img_t = torch.from_numpy(img_np).permute(2,0,1).float() / 255.

        lang_t = torch.tensor(lang_vec, dtype=torch.float32)

        if self.return_numpy:
            return img_t, label, lang_t, img_np   # raw np for sign crops
        return img_t, label, lang_t


def stratified_split(dataset, val_fraction=0.20, seed=42):
    labels  = [s[1] for s in dataset.samples]
    indices = list(range(len(labels)))
    train_idx, val_idx = train_test_split(
        indices, test_size=val_fraction, stratify=labels, random_state=seed
    )
    return train_idx, val_idx


full_train_ds = StreetviewDataset(TRAIN_DIR, transform=None, return_numpy=False)
NUM_CLASSES   = len(full_train_ds.class_to_idx)
print(f'Countries found : {NUM_CLASSES}')
print(f'Total train imgs: {len(full_train_ds)}')
print('Classes:', sorted(full_train_ds.class_to_idx.keys()))


## 8 · Class-balanced weighted sampler

Checks the per-country count and upsamples rare countries so every class contributes equally per epoch.

In [ ]:
# ── Per-class image counts
label_counts = Counter(s[1] for s in full_train_ds.samples)
print('Images per country:')
for idx in sorted(label_counts):
    print(f'  {full_train_ds.idx_to_class[idx]:30s} {label_counts[idx]:5d}')

# ── Build sample weights (inverse frequency)
sample_weights = [1.0 / label_counts[s[1]] for s in full_train_ds.samples]

# ── Identify hard pairs (common confusions) — we'll revisit after first run
# For now over-sample visually similar country clusters by 2×
HARD_PAIRS_EXTRA = {
    # fmt: country_name → extra multiplier (applied on top of inverse-freq)
    'Austria': 1.5, 'Germany': 1.5,
    'Slovakia': 1.5, 'Czech Republic': 1.5, 'Czechia': 1.5,
    'Serbia': 1.5, 'Bulgaria': 1.5,
    'Norway': 1.5, 'Sweden': 1.5, 'Finland': 1.5,
}
for i, (path, label, _) in enumerate(full_train_ds.samples):
    cname = full_train_ds.idx_to_class[label]
    if cname in HARD_PAIRS_EXTRA:
        sample_weights[i] *= HARD_PAIRS_EXTRA[cname]

print(f'\nSample weight range: {min(sample_weights):.6f} – {max(sample_weights):.4f}')


## 9 · Train / val splits and data loaders

In [ ]:
class TransformSubset(Dataset):
    """Wraps a Subset and applies albumentations transform at access time."""

    def __init__(self, subset, transform, return_numpy=False):
        self.subset       = subset
        self.transform    = transform
        self.return_numpy = return_numpy

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label, lang_vec = self.subset.dataset.samples[self.subset.indices[idx]]
        img_pil = Image.open(path).convert('RGB')
        img_np  = np.array(img_pil)

        if self.transform:
            img_t = self.transform(image=img_np)['image']
        else:
            img_t = torch.from_numpy(img_np).permute(2,0,1).float() / 255.

        lang_t = torch.tensor(lang_vec, dtype=torch.float32)

        if self.return_numpy:
            return img_t, label, lang_t, img_np
        return img_t, label, lang_t


train_idx, val_idx = stratified_split(full_train_ds, val_fraction=VAL_SPLIT, seed=SEED)

train_subset = Subset(full_train_ds, train_idx)
val_subset   = Subset(full_train_ds, val_idx)

train_ds = TransformSubset(train_subset, train_tf, return_numpy=True)  # np for sign crops
val_ds   = TransformSubset(val_subset,   val_tf,   return_numpy=False)

# ── Weighted sampler for training only
train_sample_weights = [sample_weights[i] for i in train_idx]
sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True
)

_lkw = dict(
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_lkw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **_lkw)

steps_per_epoch = len(train_loader)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Steps/epoch: {steps_per_epoch}')


## 10 · Model — EfficientNet-B3 with multi-task heads

Three output heads share the same backbone:
1. **Country head** (primary, 70% of loss) — final country prediction
2. **Sign-crop head** (aux, 30%) — country prediction from sign crops
3. **Language-group head** (aux, 15%) — predicts Cyrillic / Greek / Latin

Script features (3-dim one-hot) are **concatenated** into the country head after the pooling layer.

In [ ]:
class GeoClassifier(nn.Module):
    def __init__(self, num_classes, num_lang_groups=NUM_LANG_FEATURES):
        super().__init__()
        # ── EfficientNet-B3 backbone (handles 384×384 natively)
        base = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        # features = everything except the classifier
        self.backbone = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)

        feat_dim = 1536   # EfficientNet-B3 output channels

        # ── Country head (feat + script_vec → country)
        self.country_head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim + num_lang_groups, 768),
            nn.SiLU(),
            nn.Dropout(p=0.25),
            nn.Linear(768, num_classes),
        )

        # ── Sign-crop auxiliary head (same feat_dim, no script concat)
        self.sign_head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 512),
            nn.SiLU(),
            nn.Dropout(p=0.25),
            nn.Linear(512, num_classes),
        )

        # ── Language-group auxiliary head (3 classes: Latin/Cyrillic/Greek)
        self.lang_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(feat_dim, 128),
            nn.SiLU(),
            nn.Linear(128, num_lang_groups),
        )

    def _extract(self, x):
        """Run backbone + pool, return (B, feat_dim) vector."""

        f = self.backbone(x)
        f = self.pool(f)
        return f.flatten(1)

    def forward(self, x, lang_vec, sign_crops=None):
        """
        x         : (B, 3, H, W) full images
        lang_vec  : (B, 3) script one-hot
        sign_crops: (B*n, 3, 160, 160) or None — only passed during training
        Returns dict of logits.
        """
        feat = self._extract(x)

        # Country logits: concatenate script feature
        country_logits = self.country_head(torch.cat([feat, lang_vec], dim=1))

        # Language-group logits
        lang_logits = self.lang_head(feat)

        out = {'country': country_logits, 'lang': lang_logits}

        # Sign-crop logits (training only)
        if sign_crops is not None:
            sign_feat = self._extract(sign_crops)
            # Average crops per image: sign_crops is (B*n_crops, feat)
            # We treat each crop independently — loss averages across all
            out['sign'] = self.sign_head(sign_feat)

        return out


def set_backbone_grad(model, requires_grad: bool):
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    for name, param in m.named_parameters():
        if not name.startswith(('country_head', 'sign_head', 'lang_head')):
            param.requires_grad = requires_grad


def get_state_dict(model):
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    return m.state_dict()


model = GeoClassifier(NUM_CLASSES).to(DEVICE)
set_backbone_grad(model, requires_grad=False)
print('Backbone frozen for warm-up.')

try:
    model = torch.compile(model)
    print('torch.compile() applied.')
except Exception as e:
    print(f'torch.compile skipped: {e}')

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,}  |  Trainable (heads): {trainable:,}')


## 11 · Loss, optimiser, scheduler

In [ ]:
ce_country = nn.CrossEntropyLoss(label_smoothing=0.1)
ce_lang    = nn.CrossEntropyLoss(label_smoothing=0.05)
ce_sign    = nn.CrossEntropyLoss(label_smoothing=0.1)


def multi_task_loss(out, labels, lang_targets):
    """
    Combine country loss + sign-crop aux loss + language-group aux loss.
    lang_targets: (B,) integer class indices (0=Latin, 1=Cyrillic, 2=Greek)
    """
    L_country = ce_country(out['country'], labels)
    L_lang    = ce_lang(out['lang'], lang_targets)

    loss = L_country + LAMBDA_LANG * L_lang

    if 'sign' in out:
        # sign logits: (B*n_crops, num_classes)
        # repeat labels for each crop
        n_crops  = out['sign'].size(0) // labels.size(0)
        sign_lbl = labels.repeat_interleave(n_crops)
        L_sign   = ce_sign(out['sign'], sign_lbl)
        loss     = loss + LAMBDA_SIGN * L_sign

    return loss, L_country


def make_optimizer_and_scheduler(model, lr, epochs, steps_per_epoch):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr,
        steps_per_epoch=steps_per_epoch,
        epochs=epochs,
        pct_start=0.3,
    )
    return opt, sched


optimizer, scheduler = make_optimizer_and_scheduler(
    model, LR_HEAD, FREEZE_EPOCHS, steps_per_epoch
)
print(f'Phase-1 OneCycleLR: max_lr={LR_HEAD}, epochs={FREEZE_EPOCHS}')


## 12 · Training loop

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
GRAD_ACCUM   = 2       # effective batch = 32 * 2 = 64
LOG_STEPS    = 20      # print intra-epoch update every N steps (~30-60 sec on T4)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc    = 0.0
_phase2_started = False

try:
    from tqdm.notebook import tqdm as tqdm_nb
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print('tqdm not found — falling back to plain print progress.')


def run_train_epoch(loader, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    steps     = len(loader)
    pbar      = tqdm_nb(loader, desc=f'Ep {epoch:>3} train', leave=False,
                        unit='batch') if HAS_TQDM else loader

    for step, batch in enumerate(pbar):
        if len(batch) == 4:
            imgs, labels, lang_vecs, imgs_np = batch
            imgs_np_list = [imgs_np[i].numpy() if torch.is_tensor(imgs_np[i])
                            else imgs_np[i] for i in range(len(imgs_np))]
            sign_crops = extract_sign_crops(imgs_np_list, sign_tf, n_crops=2)
            sign_crops = sign_crops.to(DEVICE, non_blocking=True)
        else:
            imgs, labels, lang_vecs = batch
            sign_crops = None

        imgs      = imgs.to(DEVICE, non_blocking=True)
        labels    = labels.to(DEVICE, non_blocking=True)
        lang_vecs = lang_vecs.to(DEVICE, non_blocking=True)
        lang_idx  = lang_vecs.argmax(dim=1)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            out  = model(imgs, lang_vecs, sign_crops=sign_crops)
            loss, L_country = multi_task_loss(out, labels, lang_idx)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        batch_loss = L_country.item()
        batch_acc  = (out['country'].argmax(1) == labels).float().mean().item()
        total_loss += batch_loss * imgs.size(0)
        correct    += (out['country'].argmax(1) == labels).sum().item()
        total      += imgs.size(0)

        # ── Live tqdm bar update
        if HAS_TQDM:
            pbar.set_postfix(
                loss=f'{batch_loss:.4f}',
                acc=f'{batch_acc:.2%}',
                lr=f'{optimizer.param_groups[0]["lr"]:.1e}',
            )
        # ── Fallback step-level print (every LOG_STEPS steps)
        elif (step + 1) % LOG_STEPS == 0:
            run_acc = correct / total
            run_loss = total_loss / total
            lr_now = optimizer.param_groups[0]["lr"]
            print(f'  step {step+1:>4}/{steps}  '
                  f'loss={run_loss:.4f}  acc={run_acc:.2%}  lr={lr_now:.2e}')

    return total_loss / total, correct / total


def run_val_epoch(loader, epoch):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm_nb(loader, desc=f'Ep {epoch:>3} val  ', leave=False,
                   unit='batch') if HAS_TQDM else loader

    with torch.no_grad():
        for batch in pbar:
            imgs, labels, lang_vecs = batch[:3]
            imgs      = imgs.to(DEVICE, non_blocking=True)
            labels    = labels.to(DEVICE, non_blocking=True)
            lang_vecs = lang_vecs.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                out  = model(imgs, lang_vecs, sign_crops=None)
                loss = ce_country(out['country'], labels)

            total_loss += loss.item() * imgs.size(0)
            correct    += (out['country'].argmax(1) == labels).sum().item()
            total      += imgs.size(0)

            if HAS_TQDM:
                pbar.set_postfix(
                    loss=f'{loss.item():.4f}',
                    acc=f'{(out["country"].argmax(1)==labels).float().mean().item():.2%}',
                )

    return total_loss / total, correct / total


# ── Epoch loop ────────────────────────────────────────────────────────────────
print(f"{'Ep':>4} | {'Phase':>6} | {'TrLoss':>8} | {'TrAcc':>7} | "
      f"{'VlLoss':>8} | {'VlAcc':>7} | {'LR':>9} | {'Min':>5} | {'Best':>7}")
print('-' * 82)

for epoch in range(1, NUM_EPOCHS + 1):

    if epoch == FREEZE_EPOCHS + 1 and not _phase2_started:
        _phase2_started = True
        set_backbone_grad(model, requires_grad=True)
        remaining = NUM_EPOCHS - FREEZE_EPOCHS
        optimizer, scheduler = make_optimizer_and_scheduler(
            model, LR_FULL, remaining, steps_per_epoch
        )
        print(f'  → Backbone unfrozen. Phase-2: max_lr={LR_FULL}, epochs={remaining}')

    phase = 'freeze' if epoch <= FREEZE_EPOCHS else 'full'
    t0 = time.time()

    tr_loss, tr_acc = run_train_epoch(train_loader, epoch)
    vl_loss, vl_acc = run_val_epoch(val_loader,   epoch)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    improved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(get_state_dict(model), 'best_model.pth')
        improved = '✓ saved'

    lr_now  = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    print(f'{epoch:>4} | {phase:>6} | {tr_loss:>8.4f} | {tr_acc:>6.2%} | '
          f'{vl_loss:>8.4f} | {vl_acc:>6.2%} | {lr_now:>9.2e} | '
          f'{elapsed/60:>4.1f} | {best_val_acc:>6.2%} {improved}')

print(f'\nBest validation accuracy: {best_val_acc:.2%}')


## 13 · Training curves

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', lw=2)
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   lw=2, ls='--')
axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], label='Train Acc', lw=2)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   label='Val Acc',   lw=2, ls='--')
axes[1].set_title('Accuracy Curve'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 + Multi-Task — European Streetview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')


## 13b · Precompute & cache OCR script vectors

OCR is ~10× the cost of a classification pass, so running it on every evaluation is wasteful. This cell runs PaddleOCR over the dataset **once**, stores the 16-dim script vectors keyed by image path to Drive, and loads from cache on every subsequent run.

Run this once per dataset. It skips images already in the cache, so it's safe to re-run (e.g. if a session disconnects partway).

In [ ]:
import json, hashlib
from tqdm.notebook import tqdm as tqdm_nb

# ── Cache location on Drive (survives session restarts)
_CACHE_DIR  = '/content/drive/MyDrive/GeoGussrCheat/ocr_cache'
os.makedirs(_CACHE_DIR, exist_ok=True)
_CACHE_FILE = f'{_CACHE_DIR}/script_vectors_v3.json'


def _cache_key(path):
    """Stable key from the relative path (country/filename)."""
    parts = path.replace('\\', '/').split('/')
    return '/'.join(parts[-2:])   # e.g. 'France/img_001.jpg'


def load_script_cache():
    if os.path.exists(_CACHE_FILE):
        with open(_CACHE_FILE) as f:
            raw = json.load(f)
        # JSON stores lists — convert back to np arrays
        return {k: np.array(v, dtype=np.float32) for k, v in raw.items()}
    return {}


def save_script_cache(cache):
    serialisable = {k: v.tolist() for k, v in cache.items()}
    # Atomic write: temp then rename
    tmp = _CACHE_FILE + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(serialisable, f)
    os.replace(tmp, _CACHE_FILE)


def precompute_script_vectors(image_paths, save_every=200):
    """
    Run OCR over image_paths, caching results to Drive.
    Skips paths already cached. Saves incrementally so a disconnect
    doesn't lose progress.
    """
    cache = load_script_cache()
    print(f'Loaded cache with {len(cache):,} existing entries.')

    todo = [p for p in image_paths if _cache_key(p) not in cache]
    print(f'Need to OCR {len(todo):,} new images '
          f'({len(image_paths) - len(todo):,} already cached).')

    if not todo:
        print('Nothing to do — cache is complete.')
        return cache

    # Warm up the OCR engine
    get_ocr_engine()

    t0 = time.time()
    for i, path in enumerate(tqdm_nb(todo, desc='OCR', unit='img')):
        try:
            vec = detect_script_from_image(path)
        except Exception as e:
            # On any OCR failure, default to plain Latin so we never crash a run
            vec = np.zeros(NUM_LANG_FEATURES, dtype=np.float32)
            vec[15] = 1.0
        cache[_cache_key(path)] = vec

        if (i + 1) % save_every == 0:
            save_script_cache(cache)

    save_script_cache(cache)
    elapsed = time.time() - t0
    rate = len(todo) / max(elapsed, 1e-6)
    print(f'\nDone. OCR\'d {len(todo):,} images in {elapsed/60:.1f} min '
          f'({rate:.1f} img/s). Cache now has {len(cache):,} entries.')
    return cache


def cached_script_vec(path, cache, fallback_country=None):
    """
    Look up a precomputed vector. Falls back to ground-truth multihot
    (if country known) or plain Latin if not in cache.
    """
    key = _cache_key(path)
    if key in cache:
        return cache[key]
    if fallback_country is not None:
        return lang_multihot(fallback_country)
    v = np.zeros(NUM_LANG_FEATURES, dtype=np.float32); v[15] = 1.0
    return v


# ── Build the cache for the test set (and optionally train set)
_test_paths = [p for p, _, _ in test_ds.samples] if 'test_ds' in dir() else []
if not _test_paths:
    # test_ds not built yet — build a lightweight path list directly
    _test_paths = []
    for country in sorted(os.listdir(TEST_DIR)):
        cdir = os.path.join(TEST_DIR, country)
        if os.path.isdir(cdir):
            for fn in os.listdir(cdir):
                if os.path.splitext(fn)[1].lower() in {'.jpg','.jpeg','.png','.webp'}:
                    _test_paths.append(os.path.join(cdir, fn))

print(f'Test images to cache: {len(_test_paths):,}')
script_cache = precompute_script_vectors(_test_paths)

# ── Optional: also cache training set (uncomment — adds significant time)
# _train_paths = [s[0] for s in full_train_ds.samples]
# script_cache = precompute_script_vectors(_train_paths)


## 14 · Evaluate on testEurope with Test-Time Augmentation (TTA)

At inference we run **5 crops + horizontal flip = 10 forward passes** per image and average the softmax. This is a free ~1–3% accuracy boost.

In [ ]:
def build_tta_transforms(img_size=384):
    """5 deterministic crops (corners + centre) × 2 (original + hflip)."""

    base_size = int(img_size * 1.15)
    crops = [
        transforms.Compose([
            transforms.Resize(base_size),
            transforms.CenterCrop(img_size),
        ]),
        transforms.Compose([
            transforms.Resize(base_size),
            transforms.FiveCrop(img_size),
        ]),
    ]
    norm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return norm


# ── Load best checkpoint
eval_model = GeoClassifier(NUM_CLASSES).to(DEVICE)
eval_model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
eval_model.eval()
print('Best model loaded.')

# ── Build test dataset (no augmentation)
test_ds = StreetviewDataset(
    TEST_DIR, transform=val_tf,
    class_to_idx=full_train_ds.class_to_idx
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
    persistent_workers=NUM_WORKERS > 0,
)
print(f'Test images: {len(test_ds):,}')

# ── TTA inference using CACHED OCR script vectors
# use_cached_ocr=True : honest evaluation — script comes from OCR on the pixels,
#                       not the folder name. Requires Cell 13b to have run.
# use_cached_ocr=False: uses ground-truth folder-name vectors (optimistic).
def tta_predict(model, loader, dataset, use_cached_ocr=True):
    all_probs, all_labels = [], []
    sample_ptr = 0
    samples = dataset.samples

    with torch.no_grad():
        for batch in loader:
            imgs, labels, gt_lang_vecs = batch[:3]
            bs = imgs.size(0)

            if use_cached_ocr:
                # Build lang vectors from the OCR cache for this batch
                batch_vecs = []
                for j in range(bs):
                    path = samples[sample_ptr + j][0]
                    batch_vecs.append(cached_script_vec(path, script_cache))
                lang_vecs = torch.tensor(np.stack(batch_vecs), dtype=torch.float32)
            else:
                lang_vecs = gt_lang_vecs

            sample_ptr += bs
            imgs      = imgs.to(DEVICE, non_blocking=True)
            lang_vecs = lang_vecs.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                logits1 = model(imgs, lang_vecs)['country']
                logits2 = model(imgs.flip(-1), lang_vecs)['country']

            probs = (torch.softmax(logits1, 1) + torch.softmax(logits2, 1)) / 2
            all_probs.append(probs.cpu())
            all_labels.extend(labels.numpy())

    all_probs = torch.cat(all_probs, 0)
    all_preds = all_probs.argmax(1).numpy()
    return all_preds, np.array(all_labels), all_probs.numpy()


# NOTE: test_loader must have shuffle=False so sample order matches dataset.samples
all_preds, all_labels, all_probs = tta_predict(eval_model, test_loader, test_ds, use_cached_ocr=True)
test_acc = np.mean(all_preds == all_labels)
print(f'\nTest Accuracy (TTA): {test_acc:.2%}')

# Top-3 accuracy
top3_correct = sum(
    all_labels[i] in all_probs[i].argsort()[-3:]
    for i in range(len(all_labels))
)
print(f'Top-3 Accuracy     : {top3_correct / len(all_labels):.2%}')


## 15 · Per-country classification report

In [ ]:
class_names = [full_train_ds.idx_to_class[i] for i in range(NUM_CLASSES)]
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))


## 16 · Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig_h = max(12, NUM_CLASSES // 2)
plt.figure(figsize=(fig_h + 2, fig_h))
sns.heatmap(
    cm, annot=(NUM_CLASSES <= 20),
    fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title('Confusion Matrix — testEurope', fontsize=14)
plt.ylabel('True Country'); plt.xlabel('Predicted Country')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png')


## 17 · Hard-pair analysis

Identify the worst-confused country pairs so you can over-sample them in the next training run or add them to `HARD_PAIRS_EXTRA`.

In [ ]:
# ── Top-20 most confused (off-diagonal) pairs
cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)   # zero the diagonal (correct predictions)

pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_copy[i, j] > 0:
            pairs.append((cm_copy[i, j], class_names[i], class_names[j]))

pairs.sort(reverse=True)
print('Top-20 confused pairs (true → predicted):')
print(f'  {"True":25s}  {"Predicted":25s}  Errors')
print('  ' + '-'*60)
for count, true_c, pred_c in pairs[:20]:
    print(f'  {true_c:25s}  {pred_c:25s}  {count:5d}')

print()
print('Add high-error countries to HARD_PAIRS_EXTRA in Cell 8 and retrain.')


## 18 · GradCAM — what is the model looking at?

This cell visualises which parts of each image the model attends to. If it's looking at sky or cars instead of signs and markings, increase `LAMBDA_SIGN` and retrain.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ── Wrap eval_model for GradCAM (needs a single tensor output)
class GradCAMWrapper(nn.Module):
    def __init__(self, model, lang_vec):
        super().__init__()
        self.model    = model
        self.lang_vec = lang_vec  # fixed script vec for visualisation

    def forward(self, x):
        lv = self.lang_vec.expand(x.size(0), -1).to(x.device)
        return self.model(x, lv)['country']


def visualise_gradcam(image_paths, eval_model, class_to_idx, idx_to_class,
                      n_images=6, country_filter=None):
    """
    image_paths : list of file paths
    country_filter : if set, only show images from this country
    """
    import random as rnd

    # Pick images
    if country_filter:
        paths = [p for p in image_paths if country_filter.lower() in p.lower()]
    else:
        paths = image_paths
    paths = rnd.sample(paths, min(n_images, len(paths)))

    # neutral Latin script vec
    lang_vec = torch.zeros(1, 3); lang_vec[0, 0] = 1.0

    wrapped = GradCAMWrapper(eval_model, lang_vec)
    # EfficientNet-B3: last conv block
    target_layers = [eval_model.backbone[-1] if hasattr(eval_model, 'backbone')
                     else eval_model._orig_mod.backbone[-1]]

    cam = GradCAM(model=wrapped, target_layers=target_layers)

    fig, axes = plt.subplots(len(paths), 2, figsize=(10, 4 * len(paths)))
    if len(paths) == 1:
        axes = [axes]

    for ax_row, path in zip(axes, paths):
        img_pil = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        img_np  = np.array(img_pil) / 255.0
        img_t   = val_tf(image=(img_np * 255).astype(np.uint8))['image'].unsqueeze(0)

        # True label
        country = os.path.basename(os.path.dirname(path))
        label_idx = class_to_idx.get(country, 0)

        grayscale_cam = cam(input_tensor=img_t,
                            targets=[ClassifierOutputTarget(label_idx)])
        cam_image = show_cam_on_image(img_np.astype(np.float32), grayscale_cam[0])

        ax_row[0].imshow(img_pil); ax_row[0].axis('off')
        ax_row[0].set_title(f'Input: {country}', fontsize=10)
        ax_row[1].imshow(cam_image); ax_row[1].axis('off')
        ax_row[1].set_title('GradCAM attention', fontsize=10)

    plt.suptitle('GradCAM — what the model attends to', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam.png', dpi=120)
    plt.show()
    print('Saved gradcam.png')
    cam.remove_hooks()


# ── Run on a random sample from testEurope
all_test_paths = [p for p, _, _ in test_ds.samples]
visualise_gradcam(
    all_test_paths, eval_model,
    full_train_ds.class_to_idx,
    full_train_ds.idx_to_class,
    n_images=6,
    # country_filter='Germany'   # uncomment to focus on one country
)


## 19 · Single-image inference with TTA

In [ ]:
def predict_image(image_path, top_k=5, use_ocr=True, explain=True):
    """
    Predict country for a single image using TTA (original + h-flip).

    use_ocr=True  : run PaddleOCR to detect script/diacritics from visible text
    use_ocr=False : assume plain Latin (faster, less accurate)
    explain=True  : print per-feature breakdown of what was detected
    """
    idx_to_class = full_train_ds.idx_to_class
    img_pil = Image.open(image_path).convert('RGB')
    img_np  = np.array(img_pil)

    # ── Full 16-dim script + diacritic detection from actual image pixels
    if use_ocr:
        # Check the OCR cache first (built in Cell 13b) to avoid re-running OCR
        _cached = None
        if 'script_cache' in globals():
            key = _cache_key(image_path)
            if key in script_cache:
                _cached = script_cache[key]
        if _cached is not None:
            script_vec = _cached
            if explain:
                print('(using cached OCR vector)')
                for i, (name, score) in enumerate(zip(FEATURE_NAMES, script_vec)):
                    if score > 0.01:
                        print(f'  {name:20s} {score:.3f}')
        elif explain:
            script_vec = explain_script_detection(img_np)
        else:
            script_vec = detect_script_from_image(img_np)
        dominant   = FEATURE_NAMES[script_vec.argmax()]
        dom_conf   = script_vec.max() * 100
        print(f'\nDominant script: {dominant} ({dom_conf:.0f}% of detected chars)')
    else:
        script_vec       = np.zeros(NUM_LANG_FEATURES, dtype=np.float32)
        script_vec[15]   = 1.0   # plain Latin
        dominant, dom_conf = 'Plain Latin (OCR off)', 100.0
        print('Script: OCR disabled — using plain Latin default')

    lv    = torch.tensor(script_vec, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    img_t = val_tf(image=img_np)['image'].unsqueeze(0).to(DEVICE)

    eval_model.eval()
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
        logits1 = eval_model(img_t,          lv)['country']   # original
        logits2 = eval_model(img_t.flip(-1), lv)['country']   # h-flip TTA
    probs = (torch.softmax(logits1, 1) + torch.softmax(logits2, 1)) / 2

    top_probs, top_idx = torch.topk(probs[0], k=top_k)
    results = [(idx_to_class[i.item()], p.item()) for i, p in zip(top_idx, top_probs)]

    # ── Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                              gridspec_kw={'width_ratios': [1, 2]})
    axes[0].imshow(img_pil)
    axes[0].axis('off')
    axes[0].set_title(f'Input\nScript: {dominant} ({dom_conf:.0f}%)', fontsize=10)

    countries   = [r[0] for r in results]
    confidences = [r[1] * 100 for r in results]
    axes[1].barh(countries[::-1], confidences[::-1], color='steelblue')
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_xlim(0, 100)
    axes[1].set_title(f'Top-{top_k} — {results[0][0]} ({results[0][1]:.1%})')
    plt.tight_layout()
    plt.show()

    return results


def predict_batch(image_paths, use_ocr=True):
    """Run predict_image on a list of paths and return a summary DataFrame."""
    import pandas as pd
    rows = []
    for path in image_paths:
        print(f'\n── {path}')
        preds = predict_image(path, top_k=3, use_ocr=use_ocr, explain=False)
        rows.append({
            'path':   path,
            'pred_1': preds[0][0], 'conf_1': f'{preds[0][1]:.1%}',
            'pred_2': preds[1][0], 'conf_2': f'{preds[1][1]:.1%}',
            'pred_3': preds[2][0], 'conf_3': f'{preds[2][1]:.1%}',
        })
    return pd.DataFrame(rows)


# Examples:
# predict_image('/content/testEurope/Greece/img_001.jpg')
# predict_image('/content/testEurope/Poland/img_005.jpg')
# predict_image('/content/testEurope/Bulgaria/img_003.jpg')
# predict_batch(glob.glob('/content/testEurope/*/*.jpg')[:10])


## 20 · Save best model + class map to Drive

In [ ]:
import shutil

_OUT = '/content/drive/MyDrive/GeoGussrCheat/models/efficientnet_b3_v3'
os.makedirs(_OUT, exist_ok=True)

shutil.copy('best_model.pth',      f'{_OUT}/best_model.pth')
shutil.copy('training_curves.png', f'{_OUT}/training_curves.png')
shutil.copy('confusion_matrix.png',f'{_OUT}/confusion_matrix.png')

# Save class mapping
with open(f'{_OUT}/class_to_idx.json', 'w') as f:
    json.dump(full_train_ds.class_to_idx, f, indent=2)

print('Saved to Drive:')
for fname in os.listdir(_OUT):
    print(f'  {_OUT}/{fname}')
